# ST-02 — Perimetre siretisation

Les etablissements non valides en phase 1.

La version precedente comportait ici une etape supplementaire, `ST-02bis`, qui
construisait un echantillon de la base SIRENE filtre sur les SIREN issus de la
sirenisation. Elle existait uniquement pour rendre les phases 2 et 3 executables
en un temps raisonnable, et imposait de lancer la sirenisation avant la
siretisation.

Cette contrainte disparait : le blocking departemental combine au scoring
vectorise et a la lecture par partition rend le filtrage inutile. Les deux
pipelines sont desormais independants et peuvent tourner dans n'importe quel
ordre.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from src.siretisation import construire_perimetre
from src.referentiel import charger_referentiel, enrichir
from src.display import afficher_tableau
from config.settings import (
    FINESS_EGE_CLEAN, ST_VALIDES_P1, ST_PERIMETRE, PROCESSED_DIR,
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ids_valides = set(pd.read_parquet(ST_VALIDES_P1)["NB_EgeId"].astype(str))
print(f"EGE valides en phase 1 : {len(ids_valides):,}")

df_ege = pd.read_parquet(FINESS_EGE_CLEAN)
df_perimetre = construire_perimetre(df_ege, ids_valides)

EGE valides en phase 1 : 60,548
EGE totaux   : 104,649
EGE à traiter: 44,101  (exclus, validés en P1 : 60,548)


## Enrichissement APE et date d'ouverture

Le code APE et l'année d'ouverture sont rattachés dès la construction du
périmètre : la phase 2 s'en sert pour le contrôle d'activité, la phase 3 les
fait en plus entrer dans la pondération du score.

Un fichier de référentiel manquant n'est pas bloquant : le critère correspondant
est neutralisé et la pondération s'adapte à ce qui reste disponible.

In [2]:
referentiel_ege = charger_referentiel("ege")
df_perimetre = enrichir(df_perimetre, "finess_norm_ege", "ege", referentiel_ege)

print()
print(f"APE renseigne   : {(df_perimetre['ape_referentiel'] != '').sum():,} / {len(df_perimetre):,}")
print(f"Annee renseignee: {df_perimetre['annee_referentiel'].notna().sum():,} / {len(df_perimetre):,}")

Referentiel EGE
   ape   Extraction_EG_codeape.csv : 174,438 lignes, 67,292 exploitables
   date  Extraction_EG_dateouverture_20260728.csv : 174,438 lignes, 170,225 exploitables
   total 170,893 structures, 67,292 avec APE, 170,225 avec annee
Enrichissement EGE sur 44,101 lignes : APE 36.4 %, annee 97.3 %

APE renseigne   : 16,048 / 44,101
Annee renseignee: 42,899 / 44,101


In [3]:
afficher_tableau(
    df_perimetre[["NB_EgeId", "NB_PmSmsseId", "TX_NumFinessEge", "siret_norm_ege",
                  "TX_NomEgeLong", "cdcommune_norm_ege", "dept_ege"]],
    f"Apercu du perimetre ({len(df_perimetre):,} lignes)")

df_perimetre.to_parquet(ST_PERIMETRE, index=False)
print(f"Sauvegarde : {ST_PERIMETRE}")

NB_EgeId,NB_PmSmsseId,TX_NumFinessEge,siret_norm_ege,TX_NomEgeLong,cdcommune_norm_ege,dept_ege
10,410,010000115,,CH INTERCOMMUNAL AIN VAL DE SAONE - PONT DE VEYLE,01306,01
14,378,010000180,26011019200044,CENTRE HOSPITALIER PUBLIC DE HAUTEVILLE - UNITE ESPERANCE,01185,01
16,378,010000214,26011019200028,CENTRE HOSPITALIER PUBLIC DE HAUTEVILLE - UNITE ALBARINE,01185,01
21,720,010001246,39971555600045,MARPA ANNEXE DE LA MARPA DE MANZIAT,01025,01
22,634,010001261,77554408300477,ESAT LES BROSSES,01433,01


Sauvegarde : /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/data/processed/siretisation_perimetre.parquet
